# NovaBank Transaction Processing 

You work as a backend developer at NovaBank, a mobile-first neobank. The engineering team needs robust functions that handle customer transactions, enforce security policies, and generate financial reports.

Your functions must handle variable arguments, enforce keyword-only parameters for security settings, avoid mutable default pitfalls, and respect proper scope rules. Poor function design has caused security vulnerabilities before. That stops now.

## Setup

Run this cell to create sample banking data for the exercises.

In [1]:
# Sample customer and transaction data
customers = [
    {"customer_id": 1001, "name": "Emma Schmidt", "account_type": "premium", "balance": 15000},
    {"customer_id": 1002, "name": "Luc Dubois", "account_type": "basic", "balance": 3500},
    {"customer_id": 1003, "name": "Sofia Rossi", "account_type": "business", "balance": 42000},
    {"customer_id": 1004, "name": "Carlos Ruiz", "account_type": "premium", "balance": 8900},
]

transactions = [
    {"transaction_id": "T001", "customer_id": 1001, "amount": 250.50, "type": "payment"},
    {"transaction_id": "T002", "customer_id": 1002, "amount": 1000, "type": "deposit"},
    {"transaction_id": "T003", "customer_id": 1001, "amount": 75.20, "type": "withdrawal"},
]

print("Banking data loaded successfully")

Banking data loaded successfully


## Task 1: Parameter ordering basics

Create a function called `process_transaction` that follows proper parameter ordering:

1. Required positional: `customer_id`, `amount`
2. Optional with default: `transaction_type="payment"`, `currency="EUR"`
3. Return a dictionary with all transaction details

Test by calling it multiple ways:
- With only required arguments
- With all arguments using positional style
- With required arguments and some keywords

In [ ]:
# Write your function here
def process_transaction(customer_id, amount, transaction_type="payment", currency="EUR"):
    transaction = {
        "customer_id": customer_id,
        "amount": amount,
        "transaction type": transaction_type,
        "currency": currency
    }
    return transaction
# Test the function
transaction1 = process_transaction(1001, 250.50)
print(f"With required only: {transaction1}")
transaction2 = process_transaction(1002, 1000, "deposit", "USD")
print(f"With all parameters: {transaction2}")
transaction3 = process_transaction(1003, 500, currency="GBP")
print(f"With mixed parameters: {transaction3}")


With required only: {'customer_id': 1001, 'amount': 250.5, 'transaction type': 'payment', 'currency': 'EUR'}
With all parameters: {'customer_id': 1002, 'amount': 1000, 'transaction type': 'deposit', 'currency': 'USD'}
With mixed parameters: {'customer_id': 1003, 'amount': 500, 'transaction type': 'payment', 'currency': 'GBP'}


## Task 2: Keyword-only parameters for security

Create a function called `authorize_withdrawal` that takes:
- Required positional: `customer_id`, `amount`
- Keyword-only (after `*`): `auth_level`, `requires_2fa=False`

The function should:
- Check if `auth_level` is at least 2 for withdrawals over 1000 EUR
- Check if `requires_2fa` is True for withdrawals over 5000 EUR
- Return `True` if authorized, raise `PermissionError` otherwise

Test with different amounts and authorization levels.

In [32]:
# Write your function here
def authorize_withdrawal(customer_id, amount, *, auth_level, requires_2fa=False): # avec * pas besoin de required fields list
   if amount>1000 and auth_level<2:
       raise PermissionError("Authorization level too low for this withdrawal amount.")
   if amount>5000 and requires_2fa==False:
       raise PermissionError("Two-factor authentication required for withdrawals over 5000.")
   return True

# Test the function
print(authorize_withdrawal(1001, 800, auth_level=1))  # Should pass
print(authorize_withdrawal(1002, 6000, auth_level=2, requires_2fa=True))  # Should pass

# try:
#     print(authorize_withdrawal(1003, 2000, auth_level=1))  # Should raise PermissionError
#     print(authorize_withdrawal(1003, 6000, requires_2fa==True))  # Should raise PermissionError
# except PermissionError as e:
#     print(f"Authorization error: {e}")
# # print(authorize_withdrawal(1003, 6000, auth_level=3))  # Should raise PermissionError

True
True


## Task 3: Using *args for variable transaction fees

Create a function called `calculate_total_fees` that takes a base amount as the first argument, then any number of fee percentages using `*args`.

The function should:
- Apply each fee percentage to the base amount
- Return the total fees as a float rounded to 2 decimals

Example: `calculate_total_fees(1000, 1.5, 0.5, 2.0)` should calculate 1.5% + 0.5% + 2.0% of 1000.

Test with different numbers of fees.

In [ ]:
# Write your function here
def calculate_total_fees(base_amount, *fees): # args is a tuple of fee percentages
    total_fees = 0
    for fee in fees:
        total_fees += base_amount * (fee / 100) 
    return total_fees

# Test the function
total_fees1 = calculate_total_fees(1000, 1.5, 0.5, 2.0)
print(f"Total fees for transaction 1: {total_fees1}")  # Should print 40.0

Total fees for transaction 1: 40.0


## Task 4: Using **kwargs for flexible transaction metadata

Create a function called `log_transaction` that takes `transaction_id` and `amount` as required parameters, then accepts any additional metadata using `**kwargs`.

The function should:
- Create a dictionary with the transaction_id and amount
- Add all kwargs to the dictionary
- Add a timestamp using the current time (you can use a placeholder string like "2024-11-07 10:30:00")
- Return the complete dictionary

Test by passing various metadata like location, device, ip_address.

In [14]:
# Write your function here
def log_transaction(transaction_id, amount, **kwargs):
    log_entry = {   
        "transaction_id": transaction_id,
        "amount": amount,
        "timestamp": "2025-11-25 18:00:00"  # Placeholder timestamp
    }
    log_entry.update(kwargs) # kwargs est un dictionnaire
    return log_entry
log_transaction_1 = log_transaction("T100", 500.0, location="Berlin", device="Android", ip_address="192.168.1.1")
print(f"Log entry 1: {log_transaction_1}")

log_transaction_2 = log_transaction("T101", 750.0, location="Madrid")
print(f"Log entry 2: {log_transaction_2}")

Log entry 1: {'transaction_id': 'T100', 'amount': 500.0, 'timestamp': '2025-11-25 18:00:00', 'location': 'Berlin', 'device': 'Android', 'ip_address': '192.168.1.1'}
Log entry 2: {'transaction_id': 'T101', 'amount': 750.0, 'timestamp': '2025-11-25 18:00:00', 'location': 'Madrid'}


## Task 5: Avoid the mutable default trap

This function has a bug. Find and fix it:

```python
def add_transaction_to_history(transaction, history=[]):
    history.append(transaction)
    return history
```

The bug: the default list is created once and shared across all calls. Fix it using `None` as the default and creating a new list inside the function.

Test by calling the corrected function multiple times and verify each call gets a fresh list.

In [ ]:
# Write the corrected function here
def add_transaction_to_history(transaction, history=[]): 
    # les listes étant mutables, il ne faut pas utiliser un objet mutable comme valeur par défaut
    # il faut une nle liste à chaque appel
    history.append(transaction)
    return history

test_bug= add_transaction_to_history("T200")
test_bug_2= add_transaction_to_history("T201")
test_bug_3= add_transaction_to_history("T203")
print(f"Transaction history after first call: {test_bug}")
print(f"Transaction history after first call: {test_bug_2}")
print(f"Transaction history after first call: {test_bug_3}")

Transaction history after first call: ['T200', 'T201', 'T203']
Transaction history after first call: ['T200', 'T201', 'T203']
Transaction history after first call: ['T200', 'T201', 'T203']


In [21]:
def add_transaction_to_history_fixed(transaction, history=None):
    if history is None:
        history = []
    history.append(transaction)
    return history
test_fixed= add_transaction_to_history_fixed("T300")
test_fixed_2= add_transaction_to_history_fixed("T301")  
test_fixed_3= add_transaction_to_history_fixed("T303")
print(f"Transaction history after first call (fixed): {test_fixed}")    
print(f"Transaction history after second call (fixed): {test_fixed_2}")    
print(f"Transaction history after third call (fixed): {test_fixed_3}")

Transaction history after first call (fixed): ['T300']
Transaction history after second call (fixed): ['T301']
Transaction history after third call (fixed): ['T303']


## Task 6: Understanding local scope

Create a function called `calculate_overdraft_fee` that:
- Takes `balance` and `withdrawal_amount` as parameters
- Creates a local variable `fee_rate = 0.05` (5% overdraft fee)
- If withdrawal would cause overdraft (balance < withdrawal_amount), calculate and return the fee
- Otherwise return 0

After the function, try to print `fee_rate` outside the function to demonstrate it only exists inside the function scope.

Test with balances of 100 and withdrawals of 150.

In [7]:
# Write your function here


## Task 7: Working with global variables carefully

Define a global variable `DAILY_TRANSFER_LIMIT = 10000` at the module level.

Create a function called `check_transfer_limit` that:
- Takes `amount` as parameter
- Accesses the global `DAILY_TRANSFER_LIMIT` (you do not need the global keyword to read it)
- Returns `True` if amount is within limit, `False` otherwise

Then create another function `update_transfer_limit` that:
- Takes `new_limit` as parameter
- Uses the `global` keyword to modify `DAILY_TRANSFER_LIMIT`
- Updates the limit and returns the new value

Test both functions.

In [8]:
# Write your code here


## Task 8: Complete canonical parameter order

Create a function called `execute_bank_transfer` that demonstrates the complete parameter order:

1. Required: `from_customer_id`, `to_customer_id`, `amount`
2. Default: `currency="EUR"`, `priority="normal"`
3. *args: `*reference_codes` (for any number of reference codes)
4. Keyword-only: `auth_level`, `requires_approval=False`
5. **kwargs: `**metadata` (for flexible metadata)

The function should return a dictionary containing all the information.

Test with a complex call using all parameter types.

In [9]:
# Write your function here


## Task 9: Build a transaction validator with nested functions

Create a function called `create_transaction_validator` that takes a `max_amount` parameter and returns a nested validator function.

The outer function should:
- Accept `max_amount` as a parameter
- Define an inner function that takes a transaction dictionary
- The inner function checks if the transaction amount exceeds max_amount
- Return the inner function

This demonstrates closure: the inner function remembers the `max_amount` from its enclosing scope.

Test by creating a validator with a 5000 limit and validating different transactions.

In [10]:
# Write your function here


## Task 10: Calculate account balance with transaction history

Create a function called `calculate_current_balance` that:
- Takes `initial_balance` and a list of transactions
- Each transaction is a dictionary with "type" ("deposit" or "withdrawal") and "amount"
- Processes transactions in order and returns the final balance
- Use proper default parameter (avoid mutable default)

Test with an initial balance of 1000 and various transactions.

In [11]:
# Write your function here


## Task 11: Build a fee calculator with variable rates

Create a function called `apply_account_fees` that:
- Takes `balance` as required parameter
- Takes `account_type` as required parameter
- Accepts any number of additional fee names and rates using `**fee_rates`

Base fees by account type:
- "basic": 5 EUR monthly
- "premium": 15 EUR monthly
- "business": 50 EUR monthly

Add any additional fees from `**fee_rates` (they should be percentage-based on the balance).

Return the total fees.

Test with: `apply_account_fees(10000, "premium", overdraft_fee=0.5, maintenance=0.1)`

In [12]:
# Write your function here


## Task 12: Demonstrate the LEGB rule

Create code that demonstrates Python's LEGB scope resolution order (Local, Enclosing, Global, Built-in):

1. Define `commission_rate = 0.05` at global level
2. Create an outer function `create_commission_calculator` that:
   - Defines `commission_rate = 0.03` (enclosing scope)
   - Creates an inner function that:
     - Defines `commission_rate = 0.01` (local scope)
     - Returns the commission on a given amount
   - Returns the inner function
3. Create another function that uses the global `commission_rate`

Test to show which `commission_rate` each function uses.

In [13]:
# Write your code here


## Task 13: Build a fraud detection function

Create a function called `detect_suspicious_transaction` that:
- Takes required: `transaction_amount`, `customer_id`
- Takes keyword-only: `location`, `device_type`, `time_since_last_transaction`
- Uses these rules to flag suspicious transactions:
  - Amount over 10000 EUR
  - OR location is "unknown"
  - OR time_since_last_transaction < 60 seconds
- Returns a dictionary with "suspicious" (boolean) and "reasons" (list of strings)

Test with various scenarios.

In [14]:
# Write your function here


## Task 14: Create a transaction processor with closure

Create a function called `create_transaction_processor` that:
- Takes `account_id` as parameter
- Maintains a transaction count in the enclosing scope using a list `[0]` (to make it mutable)
- Returns an inner function that:
  - Takes a transaction dictionary
  - Increments the transaction count
  - Returns a processed transaction with added count and account_id

Test by creating a processor and processing multiple transactions to see the count increment.

In [15]:
# Write your function here


## Task 15: Build a comprehensive account manager

Create a function called `manage_customer_account` that combines everything:

Parameters (in canonical order):
1. Required: `customer_id`, `action` ("deposit", "withdraw", "transfer")
2. Default: `amount=0`, `currency="EUR"`
3. *args: `*beneficiaries` (for transfers to multiple accounts)
4. Keyword-only: `auth_level`, `requires_verification=True`
5. **kwargs: `**additional_data`

The function should:
- Validate the action type
- Check authorization level (must be at least 1)
- For transfers, split amount among beneficiaries
- Return a complete transaction record dictionary

Test with multiple scenarios including deposits, withdrawals, and multi-beneficiary transfers.

In [16]:
# Write your function here
